# LSA64 preprocessing → 128×128 frames, split laid out

Decodes all 3200 LSA64 clips once, centre-cropped and resized to 128×128, then
writes the D9 split as `train/` and `test/`. Training is then GPU-bound rather
than CPU-bound — Kaggle gives ~4 cores, which cannot feed two T4s from raw video.

CPU-only: **costs no GPU quota**. Measured ~47 min end to end.

**The output is consumed in place, not downloaded.** The training kernel mounts
it via `kernel_sources`. This replaces the original plan of fetching the output
to the workstation and re-publishing it as a dataset (D16): that plan costed the
round-trip in gigabytes (~1.5GB, acceptable) and missed the currency that
actually matters — **file count**. `kernels output` fetches one file at a time
and managed 1 of ~228,000 in six minutes.

This kernel still authenticates with nothing, which was D16's real requirement.

Source: `justinvo277/lsa64-dataset`, the **cut** distribution (D14) — the paper
does not say which it used, and cut removes idle frames that would inflate L1
and TCD for free.

Producing `train/`+`test/` here is load-bearing, not tidiness: `FramesDataset`
silently falls back to its own random 80/20 split when `train/` is absent, giving
2560/640 instead of the paper's 2800/400 and discarding D9 without an error (D17).

In [ ]:
import subprocess, sys

r = subprocess.run(
    ["git", "clone", "-q", "--branch", "feat/m0-m1-harness", "--depth", "1",
     "https://github.com/SonLamHG/pgmm.git", "/kaggle/working/repo"],
    capture_output=True, text=True,
)
assert r.returncode == 0, r.stderr
sys.path.insert(0, "/kaggle/working/repo")
print("repo cloned")

In [ ]:
# Discover the source clips. The mount layout is /kaggle/input/datasets/<owner>/
# <slug>/... -- not the commonly documented /kaggle/input/<slug> -- so glob for
# it rather than trusting either shape.
from pathlib import Path

INPUT = Path("/kaggle/input")
mp4s = sorted(INPUT.rglob("*.mp4"))
assert len(mp4s) == 3200, f"expected 3200 clips, found {len(mp4s)}"
SRC = mp4s[0].parent.parent  # .../LSA64  (clips are grouped in per-sign dirs)
print("clips :", len(mp4s))
print("source:", SRC)

In [ ]:
# Preprocess all 3200 clips, then lay the D9 split out on disk.
#
# The layout is not cosmetic. FramesDataset checks for root_dir/train and,
# finding none, silently does its own random 80/20 split -- 2560/640 instead of
# the paper's 2800/400, discarding D9 with no error (D17). Producing train/test
# here is what makes our split the one that trains.
import shutil
import time

from pgmm.data.layout import materialise_split
from pgmm.data.lsa64_prepare import prepare_dataset

FLAT = Path("/kaggle/working/_flat")
OUT = Path("/kaggle/working/lsa64_prepared")

t0 = time.time()
index = prepare_dataset(SRC, FLAT, size=128)
dt = time.time() - t0
print(f"decoded {len(index)} clips, {sum(index.values())} frames in {dt/60:.1f} min")
print(f"        {dt/len(index):.2f}s/clip (smoke test projected 0.70)")

counts = materialise_split(FLAT, OUT, mode="random", seed=0)
print("split:", counts)
assert counts == {"train": 2800, "test": 400}, f"paper says 2800/400, got {counts}"

# the flat copy has served its purpose; it would otherwise double the output
shutil.rmtree(FLAT, ignore_errors=True)
print("flat intermediate removed")

In [ ]:
# Verify the laid-out result. Report BOTH size and file count -- the first run
# budgeted the round-trip in gigabytes and stalled on file count instead:
# `kernels output` fetches one file at a time and managed 1 of ~228,000 in six
# minutes.
import cv2

train_clips = sorted((OUT / "train").iterdir())
test_clips = sorted((OUT / "test").iterdir())
print("train clips:", len(train_clips), "(paper: 2800)")
print("test clips :", len(test_clips), "(paper: 400)")
assert len(train_clips) == 2800 and len(test_clips) == 400

frames = sorted(OUT.rglob("frame_*.jpg"))
total_bytes = sum(f.stat().st_size for f in frames)
print("frames     :", f"{len(frames):,}")
print("total size :", f"{total_bytes/2**30:.2f} GiB")
print("mean frame :", f"{total_bytes/len(frames)/1024:.1f} KiB")

img = cv2.imread(str(frames[0]))
assert img.shape == (128, 128, 3), img.shape
print("frame shape:", img.shape, "OK")

# a clip must survive the split with its frames intact
sample = train_clips[0]
print("sample clip:", sample.name, "->", len(list(sample.glob("frame_*.jpg"))), "frames")
assert len(list(sample.glob("frame_*.jpg"))) > 0

In [ ]:
# Leave the prepared tree in /kaggle/working and stop.
#
# Kaggle bundles a notebook's output into a single `_output_.zip` by itself --
# verified: this kernel's first run produced 264,831 frames and they arrived
# intact inside a 1.25 GiB `_output_.zip`, zip integrity checked. So no manual
# archiving is needed, and the widely-cited 500-file output cap does not bite
# here; the zip is one file.
#
# The training kernel reaches this via `kernel_sources`, which mounts it at
#   /kaggle/input/notebooks/snlmhong/pgmm-preprocess/_output_.zip
# and extracts it locally. Nothing transfers through the workstation, and this
# kernel still authenticates with nothing -- D16's actual requirement.
#
# The cloned repo must go: it would be published alongside the data.
import shutil
from pathlib import Path

shutil.rmtree("/kaggle/working/repo", ignore_errors=True)

kept = sorted(p.name for p in Path("/kaggle/working").iterdir())
print("kernel output root:", kept)
print("train clips:", len(list((OUT / "train").iterdir())))
print("test clips :", len(list((OUT / "test").iterdir())))